# BiLSTM Probability Training

Run this notebook from the project root. It trains a local TensorFlow/Keras BiLSTM on the cached engineered features, saves the model under `models/`, and appends training metrics to `results/strategy_results_unified.csv`.

In [10]:
# If TensorFlow is missing in this notebook environment, run once:
# %pip install "tensorflow<2.16" scikit-learn joblib pyarrow pandas numpy

from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Project root:", ROOT)

TensorFlow: 2.19.0
Project root: c:\Users\vaibh\OneDrive\Desktop\Workstation\MultiStrategyGenerator


In [11]:
from feature_engineering import cached_feature_engineer

SYMBOL = "ethusdt"
TIMEFRAME = "15m"
SEQUENCE_LENGTH = 64
EPOCHS = 25
BATCH_SIZE = 32

FEATURE_COLUMNS = [
    # ── Price action (3) ──────────────────────────────────────────────────
    "return_1",           # normalized momentum, most important single feature
    "candle_range",       # volatility context per bar
    "is_bullish",         # direction of current candle

    # ── Momentum (4) ─────────────────────────────────────────────────────
    "rsi_14",             # primary RSI, keep only one raw RSI
    "macd_hist",          # net MACD pressure, more informative than macd+signal separately
    "stoch_rsi_k",        # faster oscillator, complements RSI-14
    "wt_diff",            # WaveTrend WT1-WT2 crossover pressure, non-redundant with RSI

    # ── Trend regime (5) ──────────────────────────────────────────────────
    "supertrend_direction",  # clean +1/-1 trend label
    "ce_direction",          # Chandelier Exit +1/-1, different ATR-ratchet logic from supertrend
    "gainzy_trend",          # RSI pivot-based trend integer, -3 to +3
    "qqe_is_bull",           # QQE smoothed RSI regime
    "smc_int_is_bull",       # internal market structure bias (faster than swing)

    # ── Volatility / bands (3) ────────────────────────────────────────────
    "bb_pct",             # position within Bollinger Bands, normalized 0-1
    "squeeze",            # BB inside KC, volatility compression signal
    "atr_14",             # raw volatility level, needed for context

    # ── Volume / flow (3) ─────────────────────────────────────────────────
    "volume_ratio",       # volume vs 20-bar mean, asset-normalized already
    "mfi",                # money flow, combines price + volume
    "cmf",                # Chaikin, different weighting from MFI, low overlap

    # ── Liquidity / structure events (3) ─────────────────────────────────
    "ict_fvg_in_bull",    # price inside unmitigated bull FVG (displacement-confirmed)
    "ict_fvg_in_bear",    # price inside unmitigated bear FVG
    "lsw_bull_sweep",     # stop hunt below pivot low, bullish reversal setup

    # ── Multi-timeframe context (4) ───────────────────────────────────────
    "close_vs_1h_ema20",      # distance from 1h trend, normalized ratio
    "close_vs_4h_ema20",      # distance from 4h trend, normalized ratio
    "context_1h_return_1",    # 1h momentum context
    "context_4h_return_1",    # 4h momentum context
]

def load_csv(symbol, timeframe):
    path = DATA_DIR / f"{symbol.lower()}_{timeframe.lower()}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def add_context_features(symbol, base):
    out = base.copy().sort_values("time")
    for tf_name in ["1h", "4h"]:
        ctx_raw = load_csv(symbol, tf_name)
        ctx = cached_feature_engineer(ctx_raw, symbol, tf_name)
        if "ema_20" not in ctx.columns:
            ctx["ema_20"] = ctx["close"].ewm(span=20, adjust=False).mean()
        ctx = ctx[["time", "return_1", "ema_20"]].rename(columns={
            "return_1": f"context_{tf_name}_return_1",
            "ema_20": f"context_{tf_name}_ema20",
        }).sort_values("time")
        out = pd.merge_asof(out, ctx, on="time", direction="backward")
        out[f"close_vs_{tf_name}_ema20"] = out["close"] / out[f"context_{tf_name}_ema20"].replace(0, np.nan) - 1
    return out

raw = load_csv(SYMBOL, TIMEFRAME)
features = cached_feature_engineer(raw, SYMBOL, TIMEFRAME)
features = add_context_features(SYMBOL, features)

for col in FEATURE_COLUMNS:
    if col not in features.columns:
        features[col] = 0.0

features = features.replace([np.inf, -np.inf], np.nan).bfill().ffill().fillna(0)
features.shape

c:\Users\vaibh\anaconda3\envs\new_env\lib\site-packages\ta\trend.py:1030: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  self._psar[i] = high2
c:\Users\vaibh\OneDrive\Desktop\Workstation\MultiStrategyGenerator\feature_engineering.py:856: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ob_in_bear"]      = pd.Series(in_bear_ob,       index=idx)   # price touching bear OB
c:\Users\vaibh\OneDrive\Desktop\Workstation\MultiStrategyGenerator\feature_engineering.py:857: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` 

(35040, 254)

In [12]:
def make_sequences(frame, feature_columns, sequence_length=64):
    df = frame.copy()
    df["future_return"] = df["close"].pct_change().shift(-1)
    df["target"] = (df["future_return"] > 0).astype(int)
    df = df.dropna(subset=["target"]).reset_index(drop=True)

    values = df[feature_columns].to_numpy(dtype=np.float32)
    scaler = StandardScaler()
    values = scaler.fit_transform(values).astype(np.float32)

    xs, ys = [], []
    for end in range(sequence_length, len(df) - 1):
        xs.append(values[end - sequence_length:end])
        ys.append(int(df.loc[end, "target"]))
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.int64), scaler

X, y, scaler = make_sequences(features, FEATURE_COLUMNS, SEQUENCE_LENGTH)
split = int(len(X) * 0.8)
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Class balance:", pd.Series(y).value_counts(normalize=True).to_dict())

Train: (27980, 64, 25) Test: (6995, 64, 25)
Class balance: {1: 0.5040743388134382, 0: 0.49592566118656184}


In [13]:
tf.keras.utils.set_random_seed(42)

# model = tf.keras.Sequential([
#     tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, len(FEATURE_COLUMNS))),
#     tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=False)),
#     tf.keras.layers.Dropout(0.20),
#     tf.keras.layers.Dense(32, activation="relu"),
#     tf.keras.layers.Dense(2, activation="softmax"),
# ])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, len(FEATURE_COLUMNS))),

    # ── First LSTM layer: return sequences for stacking ───────────────────
    # Larger first layer captures longer-range dependencies across the 64-bar window
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64, return_sequences=True,
                             kernel_regularizer=tf.keras.regularizers.l2(1e-4))
    ),
    tf.keras.layers.Dropout(0.30),

    # ── Second LSTM layer: compress to final context vector ───────────────
    # Smaller second layer forces the model to distill, reduces overfitting
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(32, return_sequences=False,
                             kernel_regularizer=tf.keras.regularizers.l2(1e-4))
    ),
    tf.keras.layers.Dropout(0.30),

    # ── BatchNorm before dense: stabilizes training across 4 assets ───────
    tf.keras.layers.BatchNormalization(),

    # ── Dense head ────────────────────────────────────────────────────────
    tf.keras.layers.Dense(32, activation="relu",
                          kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(16, activation="relu"),

    # ── Output ────────────────────────────────────────────────────────────
    tf.keras.layers.Dense(2, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/25
875/875 ━━━━━━━━━━━━━━━━━━━━ 48s 46ms/step - accuracy: 0.5144 - loss: 0.7499 - val_accuracy: 0.5541 - val_loss: 0.7124
Epoch 2/25
875/875 ━━━━━━━━━━━━━━━━━━━━ 48s 55ms/step - accuracy: 0.5598 - loss: 0.7098 - val_accuracy: 0.5790 - val_loss: 0.7015
Epoch 3/25
875/875 ━━━━━━━━━━━━━━━━━━━━ 61s 69ms/step - accuracy: 0.5751 - loss: 0.6950 - val_accuracy: 0.5823 - val_loss: 0.6922
Epoch 4/25
875/875 ━━━━━━━━━━━━━━━━━━━━ 59s 67ms/step - accuracy: 0.5891 - loss: 0.6852 - val_accuracy: 0.5894 - val_loss: 0.6864
Epoch 5/25
875/875 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.5947 - loss: 0.6800 - val_accuracy: 0.5850 - val_loss: 0.6862
Epoch 6/25
875/875 ━━━━━━━━━━━━━━━━━━━━ 77s 88ms/step - accuracy: 0.5918 - loss: 0.6775 - val_accuracy: 0.5823 - val_loss: 0.6848
Epoch 7/25
875/875 ━━━━━━━━━━━━━━━━━━━━ 92s 105ms/step - accuracy: 0.5946 - loss: 0.6735 - val_accuracy: 0.5818 - val_loss: 0.6829
Epoch 8/25
875/875 ━━━━━━━━━━━━━━━━━━━━ 52s 59ms/step - accuracy: 0.6009 - loss: 0.6718 -

In [14]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

model_path = MODEL_DIR / "bilstm_probabilities.keras"
scaler_path = MODEL_DIR / "bilstm_scaler.pkl"
model.save(model_path)
with scaler_path.open("wb") as fh:
    pickle.dump(scaler, fh)

row = {
    "result_type": "ml_training",
    "symbol": SYMBOL,
    "timeframe": TIMEFRAME,
    "model_version": "bilstm-local-v1",
    "train_sequences": len(X_train),
    "test_sequences": len(X_test),
    "epochs": len(history.history["loss"]),
    "sequence_length": SEQUENCE_LENGTH,
    "test_loss": round(float(test_loss), 6),
    "test_accuracy": round(float(test_accuracy), 6),
    "weights_path": str(model_path),
    "updated_at": pd.Timestamp.utcnow().isoformat(),
}

unified = RESULTS_DIR / "strategy_results_unified.csv"
pd.DataFrame([row]).to_csv(unified, mode="a", header=not unified.exists(), index=False)

print(json.dumps(row, indent=2))

{
  "result_type": "ml_training",
  "symbol": "ethusdt",
  "timeframe": "15m",
  "model_version": "bilstm-local-v1",
  "train_sequences": 27980,
  "test_sequences": 6995,
  "epochs": 12,
  "sequence_length": 64,
  "test_loss": 0.6776,
  "test_accuracy": 0.582416,
  "weights_path": "c:\\Users\\vaibh\\OneDrive\\Desktop\\Workstation\\MultiStrategyGenerator\\models\\bilstm_probabilities.keras",
  "updated_at": "2026-05-28T05:39:12.109550+00:00"
}


In [15]:
latest = features[FEATURE_COLUMNS].tail(SEQUENCE_LENGTH).to_numpy(dtype=np.float32)
latest = scaler.transform(latest).astype(np.float32)
probs = model.predict(latest[None, :, :], verbose=0)[0]
print({"BUY probability": round(float(probs[0] * 100), 2), "SELL probability": round(float(probs[1] * 100), 2)})

{'BUY probability': 42.13, 'SELL probability': 57.87}
